# Employee promotion prediction tool

## Import the dependencies

In [175]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
url = 'https://raw.githubusercontent.com/MainakRepositor/Datasets/refs/heads/master/HR%20Data/train.csv'

## Create a dataset

In [176]:
class EmployeeDataSet(Dataset):
    def __init__(self, url, transform=None):
        super(EmployeeDataSet, self).__init__()
        # load the dataset
        self.employee_data = pd.read_csv(url)
        self.transform = transform

        # drop the missing data
        self.employee_data = self.employee_data.dropna()

        # clean the dataset | remove unnecessary columns
        self.employee_data = pd.get_dummies(self.employee_data)

        # balance sheet so its 50/50 promoted and not
        promoted = self.employee_data[self.employee_data['is_promoted'] == 1]
        not_promoted = self.employee_data[self.employee_data['is_promoted'] == 0]
        not_promoted_sample = not_promoted.sample(n=len(promoted), random_state=42)

        self.employee_data = pd.concat([promoted, not_promoted_sample])
        self.X_labels = self.employee_data.drop(columns=['is_promoted', 'employee_id'])
        self.y_target = self.employee_data['is_promoted']

        # scale the X label
        self.scaler = StandardScaler()
        self.X_scaled = self.scaler.fit_transform(self.X_labels)
        # transform X and y into tensors for pytorch
        self.X_tensor = torch.tensor(self.X_scaled, dtype=torch.float32)
        self.y_tensor = torch.tensor(self.y_target.astype(float).values, dtype=torch.float32)

    def __len__(self):
        # return size of the dataset
        return len(self.X_labels)
    def __getitem__(self, idx):
        # return idx tensor
        return self.X_tensor[idx], self.y_tensor[idx]

## Prepare data for loading

### Create dataset

In [177]:
dataset = EmployeeDataSet(url)

### Split the dataset into test and training

In [178]:
total_size = len(dataset)
train_size = int(total_size * 0.7)
test_size = total_size - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

### Hook the training and test sets into their loaders

In [179]:
# weights = [class_weights[int(label)] for label in dataset.y_target]
# sampler = WeightedRandomSampler(weights, len(weights), replacement=True)
# sampler=sampler
training_loader = DataLoader(dataset=train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(dataset=test_dataset,batch_size=32,shuffle=False)

## Create model architecture

In [180]:
class HrPromotionTool(nn.Module):
    def __init__(self, in_features_=58, h1=64, h2=32):
        super(HrPromotionTool, self).__init__()
        self.IN = nn.Linear(in_features_, h1)
        self.fc1 = nn.Linear(h1, h2)
        self.OUT = nn.Linear(h2, 1)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        # let the data pass through the neurons
        x = F.leaky_relu(self.IN(x))
        x = self.dropout(x)
        x = F.leaky_relu(self.fc1(x))
        x = self.dropout(x)
        x = self.OUT(x)
        return x

## Training loop

### The preparations

In [181]:
model = HrPromotionTool()
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) # where lr is learning rate how precise is each step of descend
        # pos_weight = torch.tensor([50.0])
        # correct_count = predicted_classes.eq(y_batch.unsqueeze(1)).sum().item()
        # correct_percent = correct_count / y_batch.size(0) * 100
        # print(f'Model got {correct_count} correct out of {y_batch.size(0)} samples with percentage {correct_percent}%')

### The loop itself

In [182]:
epochs = 5
for epoch in range(epochs):
    running_tp = 0
    running_fp = 0
    running_actually_promoted = 0
    running_predicted_positives = 0

    for X_batch, y_batch in training_loader:
        predictions = model(X_batch)
        loss = loss_fn(predictions, y_batch.unsqueeze(1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        predicted_classes = (predictions > 0).float().view(-1) # (if sth) ?'then' do this :'or' this
        tar = y_batch.float().view(-1)
        batch_tp = (predicted_classes * tar).sum().item()
        batch_actually_promoted = tar.sum().item()
        running_predicted_positives += predicted_classes.sum().item()
        print(running_predicted_positives)
        print(f'Loss: {loss.item():.4f}')

        running_tp += batch_tp
        running_actually_promoted += batch_actually_promoted
    epoch_precision = running_tp / (running_predicted_positives + 1e-7)
    epoch_recall = running_tp / (running_actually_promoted + 1e-7)
    print(f'Epoch: {epoch + 1} Recall: {epoch_recall:.6f}')



2.0
Loss: 0.7463
7.0
Loss: 0.7061
9.0
Loss: 0.6872
12.0
Loss: 0.6909
17.0
Loss: 0.7251
20.0
Loss: 0.6993
24.0
Loss: 0.6746
25.0
Loss: 0.6865
27.0
Loss: 0.6812
29.0
Loss: 0.6740
31.0
Loss: 0.6622
31.0
Loss: 0.7089
35.0
Loss: 0.6981
36.0
Loss: 0.6964
38.0
Loss: 0.6937
41.0
Loss: 0.6800
44.0
Loss: 0.6970
45.0
Loss: 0.6925
54.0
Loss: 0.6863
59.0
Loss: 0.6797
71.0
Loss: 0.6866
77.0
Loss: 0.6778
85.0
Loss: 0.6668
92.0
Loss: 0.6861
101.0
Loss: 0.6877
110.0
Loss: 0.6865
119.0
Loss: 0.6685
130.0
Loss: 0.6810
145.0
Loss: 0.6840
156.0
Loss: 0.6849
170.0
Loss: 0.6629
181.0
Loss: 0.6797
194.0
Loss: 0.6550
208.0
Loss: 0.6666
224.0
Loss: 0.6647
241.0
Loss: 0.6654
257.0
Loss: 0.6643
275.0
Loss: 0.6577
283.0
Loss: 0.6437
289.0
Loss: 0.6462
302.0
Loss: 0.6749
314.0
Loss: 0.6442
319.0
Loss: 0.6635
330.0
Loss: 0.6613
346.0
Loss: 0.6396
362.0
Loss: 0.6221
373.0
Loss: 0.6244
386.0
Loss: 0.6532
399.0
Loss: 0.6434
412.0
Loss: 0.6941
422.0
Loss: 0.6734
433.0
Loss: 0.6471
452.0
Loss: 0.6309
470.0
Loss: 0.6289
4

## Test loop

In [191]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    running_tp = 0
    running_fp = 0
    running_actually_promoted = 0
    running_predicted_positives = 0
    for X_batch, y_batch in test_loader:
        predictions = model(X_batch)
        loss = loss_fn(predictions, y_batch.unsqueeze(1))
        total += y_batch.size(0)
        predicted_classes = (predictions > 0).float()
        correct += predicted_classes.eq(y_batch.unsqueeze(1)).sum().item()

        # check if model is not lying
        predicted_classes = (predictions > 0).float().view(-1) # (if sth) ?'then' do this :'or' this
        tar = y_batch.float().view(-1)
        batch_tp = (predicted_classes * tar).sum().item()
        batch_actually_promoted = tar.sum().item()
        running_predicted_positives += predicted_classes.sum().item()
        print(running_predicted_positives)
        print(f'Loss: {loss.item():.4f}')

        running_tp += batch_tp
        running_actually_promoted += batch_actually_promoted
    epoch_precision = running_tp / (running_predicted_positives + 1e-7)
    epoch_recall = running_tp / (running_actually_promoted + 1e-7)
    print(f'Recall: {epoch_recall:.6f}')
    print(f'Model has scored {correct} correct out of {total} samples with percentage of correct predictions {100 * correct / total:.2f}%')

18.0
Loss: 0.5468
35.0
Loss: 0.3243
53.0
Loss: 0.3846
74.0
Loss: 0.5114
89.0
Loss: 0.3216
107.0
Loss: 0.5649
128.0
Loss: 0.4239
145.0
Loss: 0.4709
168.0
Loss: 0.5667
182.0
Loss: 0.4047
204.0
Loss: 0.3484
221.0
Loss: 0.3484
238.0
Loss: 0.4457
255.0
Loss: 0.4315
272.0
Loss: 0.2754
288.0
Loss: 0.3293
306.0
Loss: 0.3690
321.0
Loss: 0.2121
339.0
Loss: 0.4932
359.0
Loss: 0.2881
378.0
Loss: 0.2730
395.0
Loss: 0.4873
410.0
Loss: 0.2735
431.0
Loss: 0.4594
452.0
Loss: 0.4812
469.0
Loss: 0.2825
487.0
Loss: 0.4275
508.0
Loss: 0.3553
530.0
Loss: 0.3480
547.0
Loss: 0.3505
563.0
Loss: 0.2858
581.0
Loss: 0.4263
598.0
Loss: 0.3361
614.0
Loss: 0.3707
633.0
Loss: 0.4387
654.0
Loss: 0.4342
673.0
Loss: 0.2757
691.0
Loss: 0.4034
708.0
Loss: 0.3402
730.0
Loss: 0.4463
747.0
Loss: 0.2671
763.0
Loss: 0.3505
785.0
Loss: 0.2566
803.0
Loss: 0.5296
815.0
Loss: 0.2652
836.0
Loss: 0.4341
856.0
Loss: 0.3308
877.0
Loss: 0.2427
895.0
Loss: 0.3032
912.0
Loss: 0.2879
933.0
Loss: 0.2788
952.0
Loss: 0.4166
972.0
Loss: 0.403

## Save model

In [190]:
FILE_PATH = './versions/HrPredictT_1.0.pt'
torch.save(model.state_dict(), FILE_PATH)

# Tool

## Take user input and parse it


In [185]:
import joblib
HR_SCALER_PATH = './util/hr_scaler.joblib'

In [186]:
joblib.dump(dataset.scaler, HR_SCALER_PATH)
column_names = dataset.X_labels.columns.tolist()
joblib.dump(column_names, HR_SCALER_PATH)
print(f'Scaler has been saved to {HR_SCALER_PATH}')

Scaler has been saved to ./util/hr_scaler.joblib
